# Export Safe OMOP Feature Mapping Input

这个 notebook 只从 `07_feature_matrix.parquet` 读取 **schema / column names**，不会读取或导出任何 patient-level values，也不会导出 `person_id`、`visit_occurrence_id`、`condition_occurrence_id` 等 ID 列。

你运行后，只需要把生成的这个文件发给我即可：

```text
output/omop_mapping_safe_input.xlsx
```

它只包含 feature column names 和自动解析的 metadata 草稿，用来后续生成 `omop_price.xlsx` 风格的 OMOP mapping。

In [ ]:
from pathlib import Path
import re
import json
import pandas as pd

try:
    import pyarrow.parquet as pq
except ImportError as e:
    raise ImportError("Please install pyarrow, or run this in the same environment that generated the parquet files.") from e

# ====== Configure paths ======
OUTPUT_DIR = Path("output")
FEATURE_MATRIX_PATH = OUTPUT_DIR / "07_feature_matrix.parquet"

SAFE_XLSX_PATH = OUTPUT_DIR / "omop_mapping_safe_input.xlsx"
SAFE_CSV_PATH = OUTPUT_DIR / "omop_mapping_safe_input.csv"

assert FEATURE_MATRIX_PATH.exists(), f"Cannot find {FEATURE_MATRIX_PATH}. Please run this notebook in the parent folder of output/.

## 1. Read parquet schema only

这里只读取 parquet 的 schema，不读取任何病人行数据。

In [ ]:
# Read column names from parquet metadata/schema only; no patient-level rows are loaded.
schema = pq.ParquetFile(FEATURE_MATRIX_PATH).schema_arrow
all_columns = list(schema.names)

print(f"Total columns in feature matrix: {len(all_columns)}")
print("First 20 columns:")
print(all_columns[:20])

## 2. Remove sensitive / non-feature columns

规则：
- 去掉所有明确 ID 列：`person_id`, `visit_occurrence_id`, `condition_occurrence_id`, etc.
- 去掉 cohort/label/group 等标签列。
- 去掉疑似 patient-level 时间戳或出生日期列。
- 只保留 feature column names。

In [ ]:
EXACT_EXCLUDE = {
    "person_id", "subject_id", "patient_id", "mrn", "empi", "pat_id",
    "visit_occurrence_id", "visit_detail_id", "condition_occurrence_id",
    "measurement_id", "drug_exposure_id", "procedure_occurrence_id",
    "observation_id", "device_exposure_id", "note_id", "specimen_id",
    "provider_id", "care_site_id", "location_id",
    "label", "target", "y", "group", "cohort", "case_control",
    "is_case", "is_control", "is_psychosis", "psychosis_label",
    "birth_datetime", "birth_date", "date_of_birth", "dob",
    "condition_start_datetime", "condition_start_date", "visit_start_datetime", "visit_start_date",
    "index_date", "start_date", "end_date"
}

# Conservative exclusion: removes ID-like columns and patient-level date-like columns.
SENSITIVE_PATTERNS = [
    r"(^|_)person(_|$)",
    r"(^|_)patient(_|$)",
    r"(^|_)subject(_|$)",
    r"(^|_)mrn(_|$)",
    r"(^|_)empi(_|$)",
    r"(^|_)visit.*id($|_)",
    r"(^|_)occurrence.*id($|_)",
    r"(^|_)measurement.*id($|_)",
    r"(^|_)observation.*id($|_)",
    r"(^|_)procedure.*id($|_)",
    r"(^|_)drug.*id($|_)",
    r"(^|_)note.*id($|_)",
    r"(^|_)provider.*id($|_)",
    r"(^|_)care_site.*id($|_)",
    r"(^|_)location.*id($|_)",
    r"birth",
    r"dob",
    r"datetime$",
    r"(^|_)date$",
]

def is_sensitive_or_nonfeature_column(col: str) -> bool:
    c = col.lower().strip()
    if c in EXACT_EXCLUDE:
        return True
    return any(re.search(p, c) for p in SENSITIVE_PATTERNS)

excluded_columns = [c for c in all_columns if is_sensitive_or_nonfeature_column(c)]
feature_columns = [c for c in all_columns if c not in excluded_columns]

print(f"Excluded sensitive/non-feature columns: {len(excluded_columns)}")
print(excluded_columns)
print(f"\nRemaining feature columns: {len(feature_columns)}")
print(feature_columns[:30])

## 3. Infer mapping metadata from feature names

这一步只根据列名做启发式解析，不访问病人数据。后续我可以基于这个文件继续补全 `Field / Description / Price / url / Notes`。

In [ ]:
def infer_xn(col: str) -> str:
    """Infer x-group from column name if present, otherwise assign a broad group."""
    m = re.search(r"(?:^|[_\-/])(x\d+)(?:[_\-/]|$)", col, flags=re.IGNORECASE)
    if m:
        return m.group(1).lower()
    c = col.lower()
    if any(k in c for k in ["age", "sex", "gender", "race", "ethnicity"]):
        return "x0"
    if any(k in c for k in ["measurement", "lab", "loinc", "score", "value"]):
        return "measurement_or_score"
    if any(k in c for k in ["condition", "icd", "snomed", "disease", "diagnosis"]):
        return "condition"
    if any(k in c for k in ["drug", "med", "rxnorm", "medication"]):
        return "drug"
    if any(k in c for k in ["procedure", "cpt", "hcpcs"]):
        return "procedure"
    if any(k in c for k in ["visit", "encounter", "inpatient", "outpatient", "er", "ed"]):
        return "visit"
    return "unknown"


def infer_field_id(col: str) -> str:
    """Extract common concept/code-like identifiers from column name when possible."""
    patterns = [
        r"concept[_\-]?(?:id)?[_\-]?(\d{3,})",
        r"measurement[_\-]?concept[_\-]?id[_\-]?(\d{3,})",
        r"condition[_\-]?concept[_\-]?id[_\-]?(\d{3,})",
        r"drug[_\-]?concept[_\-]?id[_\-]?(\d{3,})",
        r"procedure[_\-]?concept[_\-]?id[_\-]?(\d{3,})",
        r"loinc[_\-]?([0-9]+[-_][0-9]+)",
        r"icd(?:9|10)?[_\-]?([a-z]\d+[a-z0-9\.]*)",
        r"snomed[_\-]?(\d{4,})",
        r"rxnorm[_\-]?(\d{3,})",
        r"cpt[_\-]?([0-9]{4,5})",
    ]
    for p in patterns:
        m = re.search(p, col, flags=re.IGNORECASE)
        if m:
            return str(m.group(1)).replace("_", "-")
    return ""


def clean_field_name(col: str) -> str:
    name = col
    # Remove common prefixes, but keep original FullColumnName separately.
    name = re.sub(r"^(feature|feat|measurement|condition|drug|procedure|visit)[_\-]+", "", name, flags=re.IGNORECASE)
    name = re.sub(r"[_\-]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

rows = []
for col in feature_columns:
    rows.append({
        "xn": infer_xn(col),
        "FieldID": infer_field_id(col),
        "Field": clean_field_name(col),
        "Description": "auto_extracted_from_feature_column_name; needs_manual_review",
        "Price": "",
        "url": "",
        "Notes": "safe_export_no_patient_values; needs_manual_review",
        "FullColumnName": col,
    })

mapping = pd.DataFrame(rows, columns=[
    "xn", "FieldID", "Field", "Description", "Price", "url", "Notes", "FullColumnName"
])

mapping.head(20)

## 4. Safety check before export

如果任何敏感列名残留，会直接报错，不会导出。

In [ ]:
# Final safety check: do not export known sensitive columns.
sensitive_left = [c for c in mapping["FullColumnName"].tolist() if is_sensitive_or_nonfeature_column(c)]
if sensitive_left:
    raise ValueError(f"Sensitive/non-feature columns still present; export stopped: {sensitive_left}")

# Additional high-risk exact strings in exported text.
high_risk_terms = ["person_id", "patient_id", "subject_id", "mrn", "empi", "visit_occurrence_id", "condition_occurrence_id"]
joined = "\n".join(mapping.astype(str).agg("|".join, axis=1).tolist()).lower()
leaked_terms = [t for t in high_risk_terms if t in joined]
if leaked_terms:
    raise ValueError(f"Potential sensitive terms detected in export; export stopped: {leaked_terms}")

print("Safety check passed.")
print(f"Rows to export: {len(mapping)}")

## 5. Export one safe file

运行后请只把下面这个文件发给我：

```text
output/omop_mapping_safe_input.xlsx
```

它不包含任何 patient ID 或 patient-level values。

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CSV backup
mapping.to_csv(SAFE_CSV_PATH, index=False)

# Excel file for sharing
with pd.ExcelWriter(SAFE_XLSX_PATH, engine="openpyxl") as writer:
    mapping.to_excel(writer, index=False, sheet_name="safe_feature_mapping")
    summary = pd.DataFrame([
        {"item": "source_file", "value": str(FEATURE_MATRIX_PATH)},
        {"item": "privacy_mode", "value": "schema_only_no_patient_rows"},
        {"item": "total_columns_in_feature_matrix", "value": len(all_columns)},
        {"item": "excluded_sensitive_or_nonfeature_columns", "value": len(excluded_columns)},
        {"item": "exported_feature_columns", "value": len(mapping)},
    ])
    summary.to_excel(writer, index=False, sheet_name="summary")

print(f"Wrote: {SAFE_XLSX_PATH}")
print(f"Backup CSV: {SAFE_CSV_PATH}")
print("Please upload only the xlsx file to ChatGPT for OMOP mapping generation.")

## Optional: inspect exported columns only

下面只显示 feature names，不显示任何病人数据。

In [ ]:
mapping[["xn", "FieldID", "Field", "FullColumnName"]].head(50)